# Lección 9.4 — Retropropagación (*backpropagation*): repartir la culpa del error

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.3 (descenso de gradiente), Lección 3.2 (función de pérdida), Lecciones 9.1–9.3 (neurona, capas y activaciones)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué hace la retropropagación: **repartir la culpa del error** entre todos los pesos de la red.
- Calcular a mano, con números, el gradiente de una neurona usando la **regla de la cadena**, y comprobarlo de dos formas.
- Entrenar una red pequeña "a mano" con numpy, sin ninguna librería de deep learning.
- Usar **autograd** de PyTorch (`requires_grad`, `.backward()`, `.grad`, `no_grad`) y escribir de memoria el **bucle de entrenamiento estándar**.
- Reconocer el error clásico de olvidar `zero_grad()`.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split

np.random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: ¿de quién es la culpa?

Una pastelería en cadena saca un pastel **demasiado salado**. El cliente se queja con el último eslabón, el decorador. Pero la culpa no es solo suya:

```
Proveedor de sal ──► Mezclador ──► Horneador ──► Decorador ──► 🎂 pastel salado 😖
      ▲                  ▲              ▲              ▲
      └──────────────────┴──── la queja viaja hacia ATRÁS ─┘
```

El gerente reparte la culpa **de atrás hacia adelante**:
1. El decorador: *"yo solo puse el betún; influyo poco en lo salado"* → recibe poca culpa.
2. El mezclador: *"yo eché la sal; influyo muchísimo"* → recibe mucha culpa.
3. Y la culpa del mezclador se reparte a su vez con el proveedor: *"¿la sal venía más concentrada de lo normal?"*

Cada persona recibe una parte de la culpa **proporcional a cuánto influyó en el resultado**. Luego cada uno ajusta un poco su forma de trabajar.

**Eso es la retropropagación:** la red hace una predicción hacia adelante, mide el error con la función de pérdida (Lección 3.2) y luego manda la "queja" **hacia atrás**, capa por capa, para que cada peso sepa cuánta culpa tiene y en qué dirección moverse.

### ✋ Ejercicio 1 — Calentamiento
En una cadena de dos pasos, el mezclador pone **el doble** de sal de lo que recibe del proveedor, y el horneador concentra el sabor **3 veces** más.

Si el proveedor manda 1 gramo extra de sal, ¿cuántos "gramos de sabor salado" de más llegan al pastel?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**6** = 2 × 3. Cada eslabón **multiplica** el efecto del anterior. Para saber cuánto influye el primer eslabón en el resultado final, **multiplicas las influencias de toda la cadena**.

Acabas de usar la **regla de la cadena**, la única idea matemática que necesita la retropropagación.
</details>

---
## 2. 📖 El concepto: sensibilidades que se multiplican

En la Lección 3.3 vimos que el descenso de gradiente necesita saber, para cada perilla (peso), **en qué dirección y con qué fuerza** moverla para bajar la pérdida. Ese número se llama **gradiente** (o derivada) y lo puedes leer como una **sensibilidad**:

> *"Si muevo este peso un poquito, ¿cuánto cambia la pérdida?"*

En una red, un peso no toca la pérdida directamente: afecta a una neurona, que afecta a la siguiente capa, que afecta a la salida, que afecta a la pérdida. Es una cadena, como la pastelería:

```
peso w ──► z = w·x + b ──► a = activación(z) ──► pérdida L
       (×x)            (×pendiente de la activación)  (×cuánto le importa a L)
```

**Regla de la cadena:** la sensibilidad de L respecto a w es el **producto** de las sensibilidades de cada eslabón.

| Paso | Nombre | Qué hace |
|---|---|---|
| ➡️ Hacia adelante (*forward pass*) | Predecir | Calcula la salida y la pérdida, guardando los valores intermedios |
| ⬅️ Hacia atrás (*backward pass*) | Repartir la culpa | Recorre la cadena al revés multiplicando sensibilidades |
| 🔧 Actualizar | Descenso de gradiente | Cada peso se mueve: `w ← w − tasa × gradiente` |

### ✋ Ejercicio 2 — Tipos de cambio
Un dólar vale 20 pesos. Un peso vale 3 fichas del casino. Si tienes 1 dólar más, ¿cuántas fichas más tienes? ¿Y si cada ficha vale 0.5 puntos de premio?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1 dólar → 20 × 3 = **60 fichas** → 60 × 0.5 = **30 puntos**. Las "tasas de cambio" de cada eslabón se multiplican. En una red, cada tasa es una derivada pequeña y fácil de calcular; la retropropagación solo las encadena.
</details>

---
## 3. 📖 Cómo funciona por dentro: una neurona, a mano

Tomemos **una sola neurona** sigmoide (Lecciones 9.1 y 9.3) con un solo dato:
- Entrada `x = 2`, peso `w = 0.5`, sesgo `b = 0.1`.
- Respuesta correcta `y = 1`.
- Pérdida: error al cuadrado, `L = (a − y)²`.

### ➡️ Paso hacia adelante

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

x, y = 2.0, 1.0          # el dato y su respuesta correcta
w, b = 0.5, 0.1          # las perillas actuales

z = w * x + b            # suma ponderada
a = sigmoide(z)          # activación (la predicción)
L = (a - y) ** 2         # pérdida

print(f"z = {z:.4f}   a = {a:.4f}   pérdida L = {L:.4f}")

La neurona predice ≈0.75 cuando debía decir 1. Hay un error. ¿Cuánta culpa tiene `w`?

### ⬅️ Paso hacia atrás: tres eslabones
| Eslabón | Pregunta | Sensibilidad |
|---|---|---|
| L ← a | Si `a` sube un poquito, ¿cuánto cambia L? | `2·(a − y)` |
| a ← z | Si `z` sube un poquito, ¿cuánto cambia `a`? | `a·(1 − a)` (pendiente de la sigmoide) |
| z ← w | Si `w` sube un poquito, ¿cuánto cambia `z`? | `x` |

### 🔮 Ejercicio 3 — ¿Hacia dónde mover `w`?
La predicción (0.75) es **menor** que la respuesta (1). Antes de ejecutar:
1. ¿El gradiente `dL/dw` será positivo o negativo?
2. Para bajar la pérdida, ¿hay que **subir** o **bajar** `w`?

✍️ *Mis predicciones:*

In [ ]:
dL_da = 2 * (a - y)        # eslabón 1
da_dz = a * (1 - a)        # eslabón 2
dz_dw = x                  # eslabón 3
dz_db = 1                  # el sesgo entra "tal cual" en z

dL_dw = dL_da * da_dz * dz_dw     # ¡regla de la cadena: multiplicar!
dL_db = dL_da * da_dz * dz_db

print(f"dL/da = {dL_da:.4f}   da/dz = {da_dz:.4f}   dz/dw = {dz_dw}")
print(f"➡️  dL/dw = {dL_dw:.4f}")
print(f"➡️  dL/db = {dL_db:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Negativo:** `dL/dw ≈ −0.187`. El primer eslabón ya es negativo (`a − y < 0`) y los otros dos son positivos.
2. Hay que **subir** `w`: el descenso de gradiente hace `w ← w − tasa × gradiente`, y restar un número negativo **suma**. Tiene sentido: subir `w` sube `z`, sube `a` y la acerca a 1.

Fíjate en que `dL/dw` es justo el doble de `dL/db`: la única diferencia es el último eslabón (`x = 2` contra `1`). **Los pesos conectados a entradas grandes reciben más culpa.** (Otra razón para escalar los datos, Lección 2.3.)
</details>

### 📖 Comprobación 1: mover la perilla de verdad
¿Cómo sabemos que no nos equivocamos de fórmula? Podemos **medir** la sensibilidad directamente: mover `w` un poquito hacia arriba y hacia abajo y ver cuánto cambia la pérdida. Se llama **diferencia finita** y es la forma de "auditar" un gradiente.

In [ ]:
def perdida(w, b):
    return (sigmoide(w * x + b) - y) ** 2

h = 1e-5                                   # "un poquito"
dL_dw_medido = (perdida(w + h, b) - perdida(w - h, b)) / (2 * h)

print(f"Regla de la cadena : {dL_dw:.6f}")
print(f"Diferencia finita  : {dL_dw_medido:.6f}")

Coinciden. Entonces, ¿por qué no calcular **siempre** los gradientes con diferencias finitas, que son tan fáciles?

### ✋ Ejercicio 4
Una red mediana tiene **1 millón** de pesos. Con diferencias finitas, ¿cuántas veces tendrías que calcular la pérdida para obtener **un** gradiente completo? ¿Y con retropropagación?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con diferencias finitas, **2 millones** de pasadas (dos por peso: `w+h` y `w−h`). Con retropropagación, **una pasada hacia adelante y una hacia atrás**, y obtienes las sensibilidades de **todos** los pesos a la vez, porque los eslabones compartidos (los de las capas finales) se calculan una sola vez y se reutilizan.

Esa eficiencia es la razón por la que se pueden entrenar redes con miles de millones de parámetros. Las diferencias finitas solo se usan para **verificar**.
</details>

### 📖 Comprobación 2: que lo calcule PyTorch
PyTorch tiene **autograd**: si marcas un tensor con `requires_grad=True`, PyTorch **anota cada operación** que haces con él (construye un *grafo computacional*). Al llamar a `.backward()`, recorre ese grafo hacia atrás aplicando la regla de la cadena por ti, y deja cada gradiente en `.grad`.

In [ ]:
w_t = torch.tensor(0.5, requires_grad=True)   # "vigila esta perilla"
b_t = torch.tensor(0.1, requires_grad=True)

L_t = (torch.sigmoid(w_t * x + b_t) - y) ** 2  # paso hacia adelante (igual que antes)
L_t.backward()                                  # paso hacia atrás: ¡una línea!

print(f"PyTorch  dL/dw = {w_t.grad.item():.6f}   dL/db = {b_t.grad.item():.6f}")
print(f"A mano   dL/dw = {dL_dw:.6f}   dL/db = {dL_db:.6f}")
print("Grafo guardado:", L_t.grad_fn)

Tres métodos, el mismo número. A partir de ahora confiaremos en autograd, pero ya sabes **qué** está calculando.

### 🔮 Ejercicio 5 — Autograd en miniatura
Para `f = x²` con `x = 3`, ¿qué valor dejará PyTorch en `x.grad`? ¿Y para `g = 5·x + 7`?

✍️ *Mis predicciones:*

In [ ]:
x_t = torch.tensor(3.0, requires_grad=True)
f = x_t ** 2              # ✏️ cambia por 5 * x_t + 7, o por x_t ** 3
f.backward()
print("gradiente:", x_t.grad.item())

<details><summary><b>👀 Ver respuesta</b></summary>

- `f = x²` → pendiente `2·x` = **6**.
- `g = 5·x + 7` → pendiente **5** (el 7 no influye: mover `x` no lo cambia).
- (`x³` → `3·x²` = 27.)

Autograd conoce la derivada de cada operación básica y las encadena. Tú solo escribes el cálculo hacia adelante.
</details>

---
## 4. 📖 Una red de dos capas, entrenada a mano 🛠️

Ahora el reto de verdad: el **XOR** (Lección 9.1), que una sola neurona no puede resolver y una capa oculta sí (Lección 9.2). Vamos a entrenarla **solo con numpy**: paso adelante, paso atrás y actualización, escritos a mano.

```
entrada (2) ──W1,b1──► capa oculta (4, tanh) ──W2,b2──► salida (1, sigmoide) ──► pérdida
             ◄──────────── la culpa viaja hacia atrás ─────────────────────────────┘
```

> 💡 No necesitas entender cada línea. Fíjate en la **estructura**: el paso atrás recorre las mismas capas que el paso adelante, pero **en orden inverso**, y cada línea es "culpa que llega × sensibilidad local".

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)

def entrenar_xor(ocultas=4, tasa=0.5, pasos=2000, semilla=1):
    rng = np.random.default_rng(semilla)
    W1, b1 = rng.normal(0, 1, (2, ocultas)), np.zeros(ocultas)
    W2, b2 = rng.normal(0, 1, (ocultas, 1)), np.zeros(1)
    historia = []
    for paso in range(pasos):
        # ➡️ ADELANTE
        h = np.tanh(X_xor @ W1 + b1)              # capa oculta
        p = sigmoide(h @ W2 + b2)                 # salida (probabilidad)
        L = -np.mean(y_xor * np.log(p) + (1 - y_xor) * np.log(1 - p))   # entropía cruzada
        historia.append(L)
        # ⬅️ ATRÁS (regla de la cadena, capa por capa)
        culpa_salida = (p - y_xor) / len(X_xor)   # sigmoide + entropía cruzada se simplifican así
        gW2, gb2 = h.T @ culpa_salida, culpa_salida.sum(0)
        culpa_oculta = (culpa_salida @ W2.T) * (1 - h ** 2)   # × pendiente de tanh
        gW1, gb1 = X_xor.T @ culpa_oculta, culpa_oculta.sum(0)
        # 🔧 ACTUALIZAR (descenso de gradiente, Lección 3.3)
        W1 -= tasa * gW1; b1 -= tasa * gb1
        W2 -= tasa * gW2; b2 -= tasa * gb2
    return p, historia

p, historia = entrenar_xor()
print("Predicciones finales:", p.ravel().round(3), " (correctas: 0 1 1 0)")
plt.plot(historia); plt.xlabel("paso"); plt.ylabel("pérdida"); plt.title("XOR entrenado a mano con numpy"); plt.show()

🎉 Una red neuronal **completa**, entrenada sin ninguna librería de deep learning. Esto es, en esencia, lo que hace PyTorch, pero a gran escala.

### 🔮 Ejercicio 6 — ¿Qué pasa si…?
Predice y luego comprueba con la celda de abajo:
1. Con **1 sola** neurona oculta, ¿aprenderá el XOR?
2. Con una tasa de aprendizaje **diminuta** (`tasa=0.01`), ¿qué forma tendrá la curva de pérdida?

✍️ *Mis predicciones:*

In [ ]:
for ocultas, tasa in [(4, 0.5), (1, 0.5), (4, 0.01)]:     # ✏️ prueba otras combinaciones
    p, historia = entrenar_xor(ocultas=ocultas, tasa=tasa)
    print(f"ocultas={ocultas}  tasa={tasa:<5} → pérdida final {historia[-1]:.3f}   predicciones {p.ravel().round(2)}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No.** Con 1 neurona oculta la pérdida se estanca en ≈0.48 (contra ≈0.002 con 4 neuronas) y la red acierta el primer caso pero le da ≈0.67 a los otros tres: no puede separarlos. Falta capacidad (Lección 9.2).
2. La curva **baja muy despacio**: tras 2000 pasos la pérdida sigue en ≈0.47 y las predicciones (≈0.27, 0.74, 0.52, 0.47) apenas empiezan a separarse. La retropropagación calcula bien la dirección, pero los pasos son tan pequeños que casi no avanza (Lección 3.4).

> 📝 Si cambias `semilla` (los pesos iniciales al azar), verás que a veces incluso la red de 4 neuronas se queda atascada a medio camino. La inicialización importa más de lo que parece: lo verás en la Lección 9.7.

La retropropagación solo dice **hacia dónde** ir. Cuánto se avanza y si la red puede llegar dependen de la tasa de aprendizaje y de la arquitectura.
</details>

---
## 5. 📖 El bucle de entrenamiento estándar de PyTorch

Con autograd, todo el paso atrás que escribimos a mano se reduce a `perdida.backward()`, y la actualización la hace un **optimizador** (`torch.optim`, Lección 9.5). Este es el bucle que verás en **todo** el resto del curso. Apréndelo como una receta:

```python
for epoca in range(EPOCAS):
    optimizador.zero_grad()          # 1. borrar la culpa de la vuelta anterior
    pred = modelo(X)                 # 2. ➡️ adelante
    perdida = criterio(pred, y)      # 3. medir el error
    perdida.backward()               # 4. ⬅️ atrás: repartir la culpa (llena .grad)
    optimizador.step()               # 5. 🔧 mover cada peso según su culpa
```

Vamos a usarlo con las dos lunas.

In [ ]:
X_np, y_np = make_moons(n_samples=500, noise=0.2, random_state=0)
X_lunas = torch.tensor(X_np, dtype=torch.float32)
y_lunas = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

def crear_red():
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

modelo = crear_red()
criterio = nn.BCEWithLogitsLoss()                       # sigmoide + entropía cruzada (Lección 9.3)
optimizador = torch.optim.SGD(modelo.parameters(), lr=0.5)

for epoca in range(300):
    optimizador.zero_grad()
    pred = modelo(X_lunas)
    perdida = criterio(pred, y_lunas)
    perdida.backward()
    optimizador.step()
    if epoca % 50 == 0 or epoca == 299:
        print(f"época {epoca:>3}  pérdida {perdida.item():.3f}")

with torch.no_grad():                                   # evaluar: no hace falta anotar el grafo
    exactitud = ((modelo(X_lunas) > 0).float() == y_lunas).float().mean()
print(f"Exactitud: {exactitud:.1%}")

### ✋ Ejercicio 7 — Ordena la receta
Un compañero escribió los pasos desordenados. ¿En qué orden van?

- A) `optimizador.step()`
- B) `perdida = criterio(modelo(X), y)`
- C) `optimizador.zero_grad()`
- D) `perdida.backward()`

Y una pregunta extra: ¿para qué sirve `torch.no_grad()` en la evaluación?

✍️ *Mi orden:*

<details><summary><b>👀 Ver respuesta</b></summary>

**C → B → D → A**: borrar culpas viejas, predecir y medir, repartir la culpa, mover los pesos. (C también puede ir al final del bucle; lo importante es que ocurra **antes** del siguiente `backward()`.)

`torch.no_grad()` le dice a PyTorch: *"no anotes las operaciones, no voy a retropropagar"*. Ahorra memoria y tiempo al evaluar o predecir. No cambia las predicciones.
</details>

---
## 6. 📖 ¿Qué pasa si olvidas `zero_grad()`?

Este es el error más clásico de PyTorch. `.backward()` **no reemplaza** el contenido de `.grad`: lo **suma** a lo que ya había. Esto es útil a veces (por ejemplo, para acumular gradientes de varios lotes pequeños), pero si lo olvidas, cada vuelta arrastra la culpa de **todas** las vueltas anteriores.

### 🔮 Ejercicio 8
Calculamos el gradiente de `x²` en `x = 3` **tres veces seguidas** sin borrar. ¿Qué valor tendrá `x.grad` al final?

✍️ *Mi predicción:*

In [ ]:
x_t = torch.tensor(3.0, requires_grad=True)
for vuelta in range(3):
    (x_t ** 2).backward()
    print(f"vuelta {vuelta + 1}: x.grad = {x_t.grad.item()}")

<details><summary><b>👀 Ver respuesta</b></summary>

**18** = 6 + 6 + 6. El gradiente correcto es 6, pero se fue acumulando. ¿Cómo afecta esto a un entrenamiento real? Compruébalo abajo.
</details>

### 🔮 Ejercicio 9 — El bug en un entrenamiento real
Entrenamos la misma red de las lunas dos veces: una con `zero_grad()` y otra **sin** él. ¿Cómo crees que se verá la curva de pérdida del entrenamiento con el bug?

✍️ *Mi predicción:*

In [ ]:
def entrenar_lunas(borrar_gradientes, lr=0.5, epocas=300):
    modelo = crear_red()
    opt = torch.optim.SGD(modelo.parameters(), lr=lr)
    historia = []
    for epoca in range(epocas):
        if borrar_gradientes:
            opt.zero_grad()
        perdida = criterio(modelo(X_lunas), y_lunas)
        perdida.backward()
        opt.step()
        historia.append(perdida.item())
    return historia

plt.plot(entrenar_lunas(True), label="con zero_grad() ✅")
plt.plot(entrenar_lunas(False), label="SIN zero_grad() 🐞")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida (escala log)"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Lo engañoso: al principio la versión con el bug **parece ir mejor** (baja más rápido, hasta ≈0.1, porque la culpa acumulada empuja como una bola cuesta abajo). Pero los gradientes acumulados siguen creciendo sin control, cada paso es más largo que el anterior, y hacia la época 150 la pérdida **explota** a valores de 5–8 y se queda saltando arriba. La versión correcta baja de forma estable hasta ≈0.19.

Moraleja: un bug no siempre se nota en las primeras épocas. Si ves una pérdida que de repente se dispara sin motivo, revisa primero esta línea (Lección 9.9).

</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida oscila o explota sin razón aparente | Olvidaste `optimizador.zero_grad()` | Ponlo al inicio de cada iteración |
| La pérdida no cambia nada | Olvidaste `optimizador.step()` (o `backward()`) | Revisa que estén los 5 pasos de la receta |
| `RuntimeError: Trying to backward through the graph a second time` | Llamaste `backward()` dos veces sobre el mismo cálculo | Recalcula la pérdida en cada iteración |
| `element 0 of tensors does not require grad` | El tensor no estaba marcado (o se calculó dentro de `no_grad`) | Usa parámetros de `nn.Module` o `requires_grad=True` |
| La memoria crece época tras época | Guardas `perdida` (con su grafo) en una lista | Guarda `perdida.item()`, que es solo un número |
| Evaluar es lento y gasta memoria | Evalúas sin `torch.no_grad()` | Envuelve la evaluación en `with torch.no_grad():` |
| Sospechas que tu gradiente está mal (código propio) | Error en las fórmulas | Compáralo con una **diferencia finita** |

### 🔮 Ejercicio 10 — ¿Cuál es el bug?
Este bucle "entrena", pero la pérdida se queda **exactamente igual** en cada época. ¿Qué falta?

```python
for epoca in range(100):
    optimizador.zero_grad()
    perdida = criterio(modelo(X), y)
    perdida.backward()
    print(perdida.item())
```

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Falta **`optimizador.step()`**. La culpa se calcula (`backward`) pero nadie mueve los pesos, así que el modelo nunca cambia y la pérdida se repite idéntica.
</details>

---
## 8. 📋 Ficha del algoritmo

| | Retropropagación (*backpropagation*) |
|---|---|
| **Qué es** | El algoritmo que calcula el gradiente de la pérdida respecto a **todos** los pesos de una red |
| **Intuición** | Repartir la culpa del error de atrás hacia adelante, en proporción a cuánto influyó cada peso |
| **Cómo funciona** | Paso adelante guardando valores intermedios → paso atrás multiplicando sensibilidades (regla de la cadena) |
| **Qué NO es** | No es el optimizador: solo dice **hacia dónde** mover cada peso. Cuánto se mueve lo decide el optimizador (Lección 9.5) |
| **Costo** | Más o menos lo mismo que ~2 pasos adelante, sin importar cuántos pesos haya |
| **En PyTorch** | `perdida.backward()` (autograd) |
| **Verificación** | Diferencias finitas en unos pocos pesos |
| **Hiperparámetros** | Ninguno propio. (Los de la actualización: tasa de aprendizaje, optimizador… Lecciones 9.5 y 9.6) |

## 9. 🧠 Resumen en 5 puntos
1. La retropropagación **reparte la culpa** del error entre todos los pesos, de la salida hacia la entrada.
2. Usa la **regla de la cadena**: la sensibilidad de la pérdida respecto a un peso es el **producto** de las sensibilidades de cada eslabón.
3. Es **eficiente**: un paso adelante y uno atrás dan el gradiente de millones de pesos a la vez.
4. En PyTorch, **autograd** lo hace solo: `requires_grad`, `.backward()`, `.grad`; y `no_grad()` para evaluar.
5. La receta de entrenamiento es siempre la misma: **`zero_grad` → adelante → pérdida → `backward` → `step`**. Olvidar `zero_grad` acumula gradientes.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Qué calcula la retropropagación?
- a) Los mejores valores de los pesos
- b) El gradiente de la pérdida respecto a cada peso
- c) La tasa de aprendizaje óptima
- d) La exactitud del modelo

**2.** En la cadena `w → z → a → L`, las sensibilidades son `dz/dw = 3`, `da/dz = 0.1` y `dL/da = −2`. ¿Cuánto vale `dL/dw`?
- a) 1.1 · b) −0.6 · c) 0.6 · d) −6

**3.** Con el `dL/dw` de la pregunta 2, el descenso de gradiente va a…
- a) bajar `w` · b) subir `w` · c) dejar `w` igual

**4.** ¿Qué hace `optimizador.zero_grad()`?
- a) Pone los pesos en cero
- b) Borra los gradientes acumulados en `.grad`
- c) Reinicia la tasa de aprendizaje
- d) Detiene el entrenamiento si la pérdida es cero

**5.** ¿Para qué sirven las diferencias finitas en deep learning?
- a) Para entrenar redes grandes más rápido
- b) Para verificar que un gradiente está bien calculado
- c) Para reemplazar a la función de pérdida
- d) Para normalizar los datos

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Solo el gradiente (la dirección y la fuerza). Encontrar buenos pesos es trabajo del descenso de gradiente, que usa ese gradiente muchas veces.
2. **b)** 3 × 0.1 × (−2) = −0.6.
3. **b)** `w ← w − tasa × (−0.6)` → `w` sube.
4. **b)** Los pesos no se tocan; solo se limpia la "culpa" acumulada.
5. **b)** Son demasiado lentas para entrenar (dos pasadas por peso), pero perfectas para auditar.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** La pendiente de la sigmoide es `a·(1 − a)`, y su valor máximo es 0.25. En una red con 10 capas sigmoides, la culpa que llega a la primera capa se multiplica (entre otras cosas) por 10 de esas pendientes. ¿Qué crees que le pasa a la culpa que llega a las primeras capas? ¿Qué consecuencia tiene para su aprendizaje?

**B.** Un colega dice: *"La retropropagación es un algoritmo de optimización, como Adam"*. ¿Qué le corregirías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Como mucho 0.25¹⁰ ≈ 0.000001: la culpa llega **casi en cero**. Las primeras capas casi no reciben señal y aprenden muy despacio o nada. Es el problema de los **gradientes que desaparecen**, que estudiarás en la Lección 9.7, y una de las razones por las que hoy se prefiere ReLU (Lección 9.3).

**B.** Son dos piezas distintas que trabajan juntas. La retropropagación **calcula** el gradiente (la culpa de cada peso). El optimizador (SGD, Adam… Lección 9.5) **decide cómo usar** ese gradiente para mover los pesos. Puedes cambiar de optimizador sin tocar la retropropagación.
</details>

### 10.3 🧩 Práctica integradora: de la culpa a los dígitos

Vas a entrenar una red que reconoce **dígitos escritos a mano** (imágenes de 8×8 píxeles), primero dando un paso de entrenamiento **a mano** y luego con la receta completa. Hazlo en 6 pasos cortos y **no avances sin revisar cada respuesta**.

**Paso 1 ▶️ — Prepara los datos**

In [ ]:
digitos = load_digits()
X_d = torch.tensor(digitos.data / 16.0, dtype=torch.float32)   # píxeles de 0..16 → 0..1 (Lección 2.3)
y_d = torch.tensor(digitos.target)
X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.25, random_state=0, stratify=y_d)

torch.manual_seed(0)
red = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
criterio_d = nn.CrossEntropyLoss()
print(f"Entrenamiento: {len(X_tr)} imágenes · prueba: {len(X_te)} · clases: 10")

**Paso 2 🔮 — La pérdida antes de aprender.** La red todavía no sabe nada, así que reparte la probabilidad más o menos por igual entre las 10 clases (1/10 cada una). La entropía cruzada de adivinar al azar entre 10 clases es `−ln(1/10)`. ¿Cuánto vale aproximadamente? Ejecuta y compara.

✍️ *Mi apuesta:*

In [ ]:
with torch.no_grad():
    perdida_inicial = criterio_d(red(X_tr), y_tr).item()
print(f"Pérdida inicial: {perdida_inicial:.3f}   ·   −ln(1/10) = {np.log(10):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

`−ln(0.1) ≈ 2.30`, y la red da ≈2.3. ✅ Es una **prueba de cordura** muy útil: si una red de 10 clases empieza con una pérdida muy distinta de 2.3, algo raro pasa (lo usarás en la Lección 9.9).
</details>

**Paso 3 ▶️ — Un paso hacia atrás, a mano.** Calcula los gradientes y mira cuánta "culpa" recibió cada capa (medida con el tamaño total del gradiente, su *norma*).

In [ ]:
red.zero_grad()
perdida = criterio_d(red(X_tr), y_tr)
perdida.backward()
for nombre, p in red.named_parameters():
    print(f"{nombre:<9} forma {str(tuple(p.shape)):<9}  tamaño del gradiente {p.grad.norm():.4f}")

✋ ¿Cuántos parámetros recibieron un gradiente? ¿Algún tensor de pesos quedó sin `.grad`?

<details><summary><b>👀 Ver respuesta</b></summary>

Los **cuatro** tensores (pesos y sesgos de las dos capas `Linear`, 64·32 + 32 + 32·10 + 10 = 2,410 números) tienen su gradiente. Con **un solo** `backward()` obtuviste la culpa de los 2,410 parámetros a la vez. Los nombres `0.` y `2.` son las posiciones de las capas dentro de `nn.Sequential` (la `1` es la ReLU, que no tiene pesos).
</details>

**Paso 4 ✏️ — Da tú el paso de descenso de gradiente.** Sin optimizador: mueve cada peso con `w ← w − tasa × gradiente`. Usamos `no_grad()` porque esta resta **no** debe formar parte del grafo.

In [ ]:
TASA = 0.5   # ✏️ prueba también 5 y 50

with torch.no_grad():
    for p in red.parameters():
        p -= TASA * p.grad
    perdida_nueva = criterio_d(red(X_tr), y_tr).item()

print(f"Pérdida antes: {perdida.item():.4f}  →  después de UN paso: {perdida_nueva:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `TASA = 0.5` la pérdida baja un poco (≈2.33 → ≈2.30): un paso en la dirección correcta. Con `TASA = 5` (vuelve a ejecutar desde el Paso 1 para empezar de cero) baja más (≈2.18). Con `TASA = 50` la pérdida **sube** (≈4.5): el paso es tan largo que te pasas del valle (Lección 3.4). El gradiente dice la dirección; la tasa decide la distancia.
</details>

**Paso 5 ▶️ — Ahora con la receta completa.** Reemplazamos el paso a mano por `torch.optim.SGD`, que hace exactamente la misma resta, y entrenamos 300 épocas.

In [ ]:
torch.manual_seed(0)
red = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
opt = torch.optim.SGD(red.parameters(), lr=0.5)

for epoca in range(300):
    opt.zero_grad()                          # 1
    perdida = criterio_d(red(X_tr), y_tr)    # 2 y 3
    perdida.backward()                       # 4
    opt.step()                               # 5

with torch.no_grad():
    exact_tr = (red(X_tr).argmax(1) == y_tr).float().mean()
    exact_te = (red(X_te).argmax(1) == y_te).float().mean()
print(f"Pérdida final {perdida.item():.3f} · exactitud entrenamiento {exact_tr:.1%} · prueba {exact_te:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

La pérdida baja de ≈2.3 a ≈0.08 y la red reconoce ≈98 % de los dígitos de entrenamiento y ≈97 % de los de prueba. Los 5 pasos de la receta, repetidos 300 veces, bastan.
</details>

**Paso 6 ✋ — Para pensar.** Mira la pequeña brecha entre entrenamiento y prueba. Con lo que sabes del Módulo 4, ¿qué te dice? ¿Qué crees que pasaría si entrenaras 3,000 épocas en vez de 300?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Una brecha de 1–2 puntos es normal: un poco de **sobreajuste** (Lección 4.4), nada grave. Con muchas más épocas la pérdida de entrenamiento seguiría bajando hacia cero y la red seguiría memorizando, mientras que la exactitud de prueba se estancaría o bajaría un poco. Para elegir cuándo parar se usa una curva de validación y *early stopping* (Lecciones 7.2 y 9.6), y para frenar la memorización, las técnicas de la Lección 9.8.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** 3Blue1Brown, "What is backpropagation really doing?" y "Backpropagation calculus" (inglés, con subtítulos). La mejor animación del tema.
- 🛠️ **Construye tu propio autograd:** Andrej Karpathy, "The spelled-out intro to neural networks and backpropagation: building micrograd" (video y repositorio *micrograd*).
- 📘 **Lectura:** apuntes del curso CS231n de Stanford, "Backpropagation, Intuitions".
- 📘 **Documentación:** PyTorch, tutorial *A Gentle Introduction to torch.autograd*.

---
⬅️ **Anterior:** [9.3 Funciones de activación: sigmoide, tanh, ReLU y familia](9.3-funciones-de-activacion.ipynb) · ➡️ **Siguiente:** [9.5 Optimizadores: SGD, Momentum, RMSProp y Adam, o cómo bajar la montaña con estilo](9.5-optimizadores.ipynb)